## 08 — Profile × reference correlation heatmaps across candidate k (n=150, Run D pseudobulk ref, paper genes)

### Aim
* Visualize, as profile × reference-pseudobulk Spearman-correlation heatmaps, the Stage 1 fits produced by the fresh `xdec_autotuned_n150/` pipeline (n=150 PanCanAtlas cohort, Run D pseudobulk reference, paper's 282-gene Table S3 list) at the k's chosen after the full k=3..20 stability sweep — not k=9 alone.
* k candidates correct two issues with picking k by stability score alone: (1) the stability metric mechanically favors low k, since a simpler factorization is trivially more reproducible across resampled subsets; (2) k must be at least as large as the reference's distinct cell-type count, or multiple reference types are forced to share one latent profile.

### Input
* `xdec_autotuned_n150/stage0/stability_scores.tsv` — full k=3..20 stability sweep
* `xdec_autotuned_n150/stage1_k{9,13,16,17}/cell_type_profiles.tsv` — Stage 1 fits at k=9 (paper's value, kept for comparison despite sitting below the reference's 10-cell-type floor) and k=13/16/17 (the best-scoring k's that actually clear that floor)
* Run D pseudobulk reference (`data/deconvolution/xdec/paper_style_ref/refs_pseudobulk_raw.tsv` + `ref_metadata_pseudobulk_raw.tsv`) and the paper's 282-gene Table S3 list (`data/deconvolution/xdec/gene_lists/xdec_chi_table_s3_genes.txt`)

### Output
* One profile × reference-pseudobulk correlation heatmap per candidate k, same visual style as `01_artifact_diagnosis.ipynb` / `06_sliding_window_paperGenes.ipynb`

### Steps
* Load the Run D pseudobulk reference and order its columns by the 10-cell-type biological ordering used throughout `paper_duplication/`
* For each candidate k, correlate every estimated profile against every reference pseudobulk column (Spearman, over shared paper-gene probes) and render as a clustermap (no clustering, fixed biological column order)

### Result
* ✅ **One compartment resolves cleanly at every k**: stromal (pancreatic stellate cell + fibroblast) always gets its own narrowly-correlating profile — k=9 Profile_8, k=13 Profile_1, k=16 Profile_4/7, k=17 Profile_6/9/13. The single best-separated signal in the whole analysis.
* ⚠️ **Immune and epithelial compartments never split further**: T cell / B lineage / myeloid leukocyte are always captured together as one lumped "immune" block across several profiles (never individually isolated); pancreatic acinar / ductal / enteroendocrine likewise stay lumped as one "epithelial" block. True at k=9 through k=17.
* ❌ **Raising k past the 10-cell-type floor does not resolve the ambiguous/broad-correlating profile — it multiplies it**: k=9 has ~2 broad profiles (1, 9); k=13 has ~2 (6, 7); k=16 has 4 (9, 11, 12, 15); k=17 has 2 (12, 16) plus several moderately-broad ones. No k eliminates broad profiles.
* ❌ **New failure mode at higher k: degenerate "dead" profiles** (near-zero correlation with every reference pseudobulk) appear and increase with k: 0 at k=9, 1 at k=13 (Profile_8), 3 at k=16 (Profile_1/5/8), 2 at k=17 (Profile_8/17). These look like NNLS overfitting once k outstrips the information content of the probe set.

### Conclusion
* Consistent with the rest of this investigation's k-sweep findings: **no tested k (9, 13, 16, or 17) resolves the ambiguous-profile artifact**, including k's chosen specifically to clear the reference's 10-cell-type floor rather than by stability score alone. Going higher within this range trades one failure mode (an ambiguous profile) for a worse mix of two (more ambiguous profiles *and* dead ones) rather than cleanly separating out every reference cell type.
* The one robust, k-independent result is that the stromal (stellate+fibroblast) compartment is reliably separable — any future work on this dataset's deconvolution should treat that as the benchmark signal, and treat immune/epithelial sub-type resolution as still unsolved at this gene-list/reference/cohort combination.


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Find progress/ from either the notebook directory or progress/ itself.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "tools/XDec_SM/python/xdec.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start from progress/ or one of its subdirectories.")
os.chdir(ROOT)

XDEC_DIR = "data/deconvolution/xdec"
NEW_RUN_DIR = "xdec_autotuned_n150"
K_VALUES = [9, 13, 16, 17]
REFERENCE_FLOOR = 10  # number of distinct cell types in the Run D reference

mpl.rcParams.update({"font.family": "sans-serif", "font.size": 8, "axes.linewidth": 0.6, "pdf.fonttype": 42})

SEQ_BLUE_STEPS = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
                  "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
seq_blue = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE_STEPS, N=256)


def read_genes(path):
    genes = Path(path).read_text().splitlines()
    return list(dict.fromkeys(g.strip() for g in genes if g.strip()))

: 

### Run D pseudobulk reference and paper gene list

Run D's pseudobulk columns are already at (replicate × cell-type) granularity (e.g. `T_cell_pb1`, `T_cell_pb2`, …), so — unlike the single-cell `refs.tsv` used in notebooks 01/02/04–06 — no additional per-patient pseudobulking step is needed before correlating against estimated profiles.

In [ ]:
refs_pb = pd.read_csv(os.path.join(XDEC_DIR, "paper_style_ref", "refs_pseudobulk_raw.tsv"), sep="\t", index_col=0)
ref_meta_pb = pd.read_csv(os.path.join(XDEC_DIR, "paper_style_ref", "ref_metadata_pseudobulk_raw.tsv"), sep="\t", index_col=0)
probe_list = read_genes(os.path.join(XDEC_DIR, "gene_lists", "xdec_chi_table_s3_genes.txt"))

if not refs_pb.index.is_unique or not refs_pb.columns.is_unique or not ref_meta_pb.index.is_unique:
    raise ValueError("Duplicate reference or metadata labels")
pb_cell_types = ref_meta_pb.loc[refs_pb.columns, "Cell_Type"]
if pb_cell_types.isna().any():
    raise ValueError("Missing cell types in Run D reference metadata")

cell_type_order = [
    "pancreatic acinar cell", "pancreatic ductal cell", "enteroendocrine cell",
    "pancreatic stellate cell", "fibroblast", "blood vessel endothelial cell", "neural cell",
    "T cell", "lymphocyte of B lineage", "myeloid leukocyte",
]
if set(cell_type_order) != set(pb_cell_types.unique()):
    raise ValueError("Cell-type order must cover the Run D reference metadata")
cell_palette = dict(zip(cell_type_order, sns.color_palette("tab10", len(cell_type_order))))

pb_order = pb_cell_types.map({t: i for i, t in enumerate(cell_type_order)}).sort_values(kind="stable").index
reference_means = refs_pb.loc[:, pb_order]

print(f"Run D pseudobulk reference: {reference_means.shape[0]} genes × {reference_means.shape[1]} pseudobulk replicates "
      f"across {len(cell_type_order)} cell types")
print(f"{len(probe_list)} paper Table S3 genes loaded")

### Per-k profile × reference correlation heatmaps

One panel per candidate k. Each row is an estimated profile (`Profile_1..Profile_k`); each column is one Run D pseudobulk replicate, colored and grouped by cell type. k=9 is annotated as sitting below the reference's 10-cell-type floor.

In [ ]:
stability_scores = pd.read_csv(os.path.join(NEW_RUN_DIR, "stage0", "stability_scores.tsv"), sep="\t", index_col=0)
figures_dir = Path("xdec_autotuned_n150/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

correlation_panels = {}

for k in K_VALUES:
    stage1_dir = os.path.join(NEW_RUN_DIR, f"stage1_k{k}")
    profiles_path = os.path.join(stage1_dir, "cell_type_profiles.tsv")
    if not Path(profiles_path).exists():
        raise FileNotFoundError(
            f"Missing {profiles_path}. Run xdec_autotuned_n150/run_pipeline.sh stage12 {k} first."
        )
    profiles = pd.read_csv(profiles_path, sep="\t", index_col=0)
    profile_order = [f"Profile_{i}" for i in range(1, k + 1)]

    genes = [g for g in probe_list if g in reference_means.index and g in profiles.index]
    if len(genes) < 3:
        raise ValueError(f"k={k}: insufficient shared genes")
    combined = pd.concat([reference_means.loc[genes], profiles.loc[genes, profile_order]], axis=1)
    correlations = combined.corr(method="spearman").loc[profile_order, pb_order]
    if not np.isfinite(correlations.to_numpy()).all():
        raise ValueError(f"k={k}: undefined correlations; check for constant expression vectors")
    correlation_panels[k] = correlations

    score = float(stability_scores.loc[k, "stability_score"]) if k in stability_scores.index else float("nan")
    floor_note = " (below the 10-cell-type floor)" if k < REFERENCE_FLOOR else ""

    grid = sns.clustermap(
        correlations, row_cluster=False, col_cluster=False,
        col_colors=pb_cell_types.loc[pb_order].map(cell_palette).to_numpy(),
        cmap=seq_blue, vmin=0.3, vmax=1.0,
        xticklabels=False, yticklabels=True, figsize=(12, max(3.0, 0.42 * k + 1)),
        cbar_kws={"label": "Spearman r", "ticks": [0, 0.25, 0.5, 0.75, 1]}, dendrogram_ratio=(0.02, 0.02),
    )
    grid.ax_heatmap.set_xlabel(f"{len(pb_order)} Run D pseudobulk replicates (sorted by cell type)")
    grid.ax_heatmap.set_ylabel("Estimated profile (fixed order)")
    grid.ax_heatmap.yaxis.set_label_position("left")
    grid.ax_heatmap.tick_params(axis="y", rotation=0)

    handles = [plt.Rectangle((0, 0), 1, 1, color=cell_palette[t]) for t in cell_type_order]
    legend = grid.ax_heatmap.legend(
        handles, cell_type_order, title="Cell type", bbox_to_anchor=(1.05, 1.02),
        loc="upper left", fontsize=7, title_fontsize=8, frameon=False,
    )
    grid.fig.suptitle(
        f"k={k}{floor_note}, stability={score:.4f}\n"
        f"n=150, Run D pseudobulk ref, paper Table S3 genes (282)",
        y=1.03, fontsize=10,
    )
    grid.fig.canvas.draw()
    bbox = legend.get_window_extent(grid.fig.canvas.get_renderer()).transformed(grid.fig.transFigure.inverted())
    grid.cax.set_position([bbox.x0, bbox.y0 - 0.20, 0.015, 0.15])
    grid.cax.yaxis.label.set_fontsize(7)

    fig_path = figures_dir / f"fig_k_candidates_k{k}_profile_reference_correlation.png"
    grid.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(grid.fig)
    print("saved", fig_path)